# 3.2. Synthetic Regression Data
D2L의 Synthetic Regression Data 장을 PyTorch 기준으로 정리함.

이 장은 선형 회귀용 가짜 데이터를 만드는 것이다.

우리가 정답을 알고있는 데이터를 만들면, 나중에 모델이 그 정답을 제대로 찾아내는지 확인할 수 있기 때문에 가짜 데이터를 사용한다.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [3]:
%matplotlib inline
import random
import torch
import matplotlib.pyplot as plt
from torch.distributions.multinomial import Multinomial
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

torch.manual_seed(0)
random.seed(0)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. Synthetic Regression Data가 무엇인가

synthetic data는 직접 만든 가짜 데이터이다.

현실 데이터는 정답 규칙을 잘 모른다. 예를 들어서 집값데이터를 보면, 집값이 정확히 어떻게 결정되는지 모른다. 면적이나 교통, 위치같은것들이 복잡하게 연관된다.

선형회귀를 처음 구현할 때부터 현실 데이터를 사용하면 모델이 안 맞는 이유가 코드 버그때문인지, 데이터가 너무 복잡해서인지, noise가 커서인지 구분하기 어렵기때문이다.

그래서 일부로 쉬운 데이터를 만들어 사용한다.

```py
y = 2 * x1 - 3.4 * x2 + 4.2 + noise # 이렇게 만들면 미리 정답을 알수 있다

true_w = [2, -3.4]
true_b = 4.2

w ≈ [2, -3.4] # 나중에 학습시킬때 이렇게 가까워지면
b ≈ 4.2       # 학습 코드가 정상적으로 작동한다고 판단할 수 있다.
```

## 2. 이 장의 목적

전체 흐름이다.
1. x를 랜덤으로 만든다.
2. 진짜 w, b를 정한다.
3. y = Xw + b + noise 로 label을 만든다.
4. 데이터를 train/validation으로 나눈다.
5. minibatch 단위로 꺼내지도록 DataLoader를 만든다.

여기에서 03_02장에 잠깐 나온 train_dataloader가 어떻게 만들었는지 볼 수 있다.

## 3. 수식 해석

D2L의 핵심 수식이다.
$$
\mathbf{y}= \mathbf{X}\mathbf{w} + b + \boldsymbol{\epsilon}
$$

D2L은 2차원 feature를 가진 예제 1000개를 표준정규분포에서 만들고, lable은 ground-truth linear function에 noise를 더해서 만든다고 설명했다. noise는 평균 0, 표준편차 0.01인 정규분포에서 뽑는다.

- X: 입력 데이터 전체
- w: 진짜 가중치
- b: 진짜 편향
- ε: noise 약간의 랜덤 오차
- y: 정답 label

예를 들어서

```py
x = [x1, x2] # feature가 2개라면 샘플 하나는 이렇게 생겼다.

w = [2, -3.4] # 가중치는 이렇게 정한다.
b = 4.2

y = 2 * x1 + (-3.4) * x2 + 4.2 + noise # 그러면 샘플 하나의 정답은 이렇게 만들어진다.

y = X @ w + b + noise # 이걸 모든 샘플에 한번에 적용하면 이렇게 된다.
```
X @ w는 모든 샘플에 대해 선형식을 한 번 계산하는 행렬곱이다.

## 4. shape으로 이해하기

D2L 코드는 feature가 2개인 데이터를 만든다.

```py
X.shape = (num_examples, 2) # 각 샘플은 숫자 2개를 가진다.

w.shape = (2000, 2) # 예를 들어 train 1000개, validation 1000개를 만들면 전체데이터 개수는 2000개이다.

w.shape = (2,) # w는 feature마다 하나씩 있으므로 길이가 2다.

w.reshape((-1, 1)) # 그런데 행렬곱을 하려면 보통 w를 세로 벡터로 바꿔야 한다.

w.shape = (2, 1) # 그러면 행렬곱은 이렇게 맞는다.

X.shape              # (2000, 2)
w.reshape(-1, 1)     # (2, 1)

X @ w                # (2000, 1)

y.shape = (2000, 1) # 결과적으로 샘플마다 하나의 정답이기 때문에 이렇게 된다.
```

## 5. PyTorch 코드로 직접 구현

D2L 원문은 `d2l.DataModule`을 상속해서 `SyntheticRegressionData` 클래스를 만든다. 핵심 코드는 `X = torch.randn(n, len(w))`, `noise = torch.randn(n, 1) * noise`, `y = X @ w + b + noise`이다.

In [9]:
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2

num_examples = 1000
num_features = len(true_w)

X = torch.randn(num_examples, num_features)

noise = torch.randn(num_examples, 1) * 0.01

y = X @ true_w.reshape(-1, 1) + true_b + noise

print("X shape:", X.shape)
print("y shape:", y.shape)

print("first X:", X[0]) # 첫 번째 샘플 feature
print("first y:", y[0]) # 샘플의 정답

X shape: torch.Size([1000, 2])
y shape: torch.Size([1000, 1])
first X: tensor([ 0.9288, -0.0344])
first y: tensor([6.1606])


## 6. noise는 왜 넣는가

noise는 현실 데이터의 불완전한것을 흉내 내는 것이다.
noise가 없으면 데이터가 너무 완벽하게 나올 것이다.

선형회귀 모델은 noise까지 완벽히 외우는게 목적이 아니라 noise가 있더라도 규칙을 찾아야한다.

## 7. D2L 클래스 코드 해석

In [ ]:
class SyntheticRegressionData(d2l.DataModule):
    """Synthetic data for linear regression."""
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000,
                 batch_size=32):
        super().__init__()
        self.save_hyperparameters()
        n = num_train + num_val
        self.X = torch.randn(n, len(w))
        noise = torch.randn(n, 1) * noise
        self.y = torch.matmul(self.X, w.reshape((-1, 1))) + b + noise